# Notebook 01 — Data Exploration
## UCI South German Credit Dataset

**Purpose:** Load, inspect and understand the dataset before any modelling.

**Dataset:** UCI South German Credit (CC BY 4.0)  
**DOI:** https://doi.org/10.24432/C5X89F


In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_preparation import (
    load_raw_data, describe_dataset, prepare_data,
    TARGET_COL, CATEGORICAL_COLS, NUMERICAL_COLS,
)

SEED = 42
np.random.seed(SEED)

%matplotlib inline
plt.rcParams.update({"figure.figsize": (10, 6), "axes.grid": True, "grid.alpha": 0.3})


## 1. Load Raw Data

In [ ]:
df = load_raw_data("../data/raw")
print(f"Shape: {df.shape}")
df.head()


## 2. Dataset Summary

In [ ]:
summary = describe_dataset(df)
for k, v in summary.items():
    print(f"{k:>20s}: {v}")


## 3. Class Distribution

Target:  recoded so **1 = bad credit** (positive class), **0 = good credit**.

In [ ]:
counts = df[TARGET_COL].value_counts().sort_index()
print(counts)
print(f"
Bad-credit rate: {df[TARGET_COL].mean():.1%}")

fig, ax = plt.subplots(figsize=(5, 4))
counts.plot.bar(ax=ax, color=["#4CAF50", "#F44336"])
ax.set_xticklabels(["Good (0)", "Bad (1)"], rotation=0)
ax.set_ylabel("Count")
ax.set_title("Class Distribution")
fig.savefig("../results/figures/class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. Numerical Variables — Descriptive Statistics

In [ ]:
df[NUMERICAL_COLS].describe().round(2)


## 5. Histograms of Numerical Variables

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()
for i, col in enumerate(NUMERICAL_COLS):
    axes[i].hist(df[col], bins=25, edgecolor="black", alpha=0.7)
    axes[i].set_title(col)
# Hide extra subplot
for j in range(len(NUMERICAL_COLS), len(axes)):
    axes[j].set_visible(False)
fig.suptitle("Numerical Variable Distributions", fontsize=14)
fig.tight_layout()
fig.savefig("../results/figures/numerical_histograms.png", dpi=150, bbox_inches="tight")
plt.show()


## 6. Categorical Variable Value Counts

In [ ]:
for col in CATEGORICAL_COLS:
    print(f"--- {col} ---")
    print(df[col].value_counts().sort_index())
    print()


## 7. Correlation Matrix (Numerical Variables)

In [ ]:
corr = df[NUMERICAL_COLS + [TARGET_COL]].corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)))
ax.set_yticks(range(len(corr)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.columns)
fig.colorbar(im, ax=ax)
ax.set_title("Correlation Matrix")
fig.tight_layout()
fig.savefig("../results/figures/correlation_matrix.png", dpi=150, bbox_inches="tight")
plt.show()


## 8. Preprocessing Pipeline Test

In [ ]:
data = prepare_data(df, test_size=0.20, random_state=SEED, scale=True)

print(f"X_train shape: {data['X_train'].shape}")
print(f"X_test  shape: {data['X_test'].shape}")
print(f"y_train distribution: good={int((data['y_train']==0).sum())}, bad={int((data['y_train']==1).sum())}")
print(f"y_test  distribution: good={int((data['y_test']==0).sum())}, bad={int((data['y_test']==1).sum())}")
print(f"
Number of features after encoding: {data['X_train'].shape[1]}")
print(f"Feature names (first 10): {data['feature_names'][:10]}")


## 9. Data Dictionary (Summary)

| Column | Type | Description |
|--------|------|-------------|
| laufkont | Categorical | Status of existing checking account |
| laufzeit | Numerical | Duration in months |
| moral | Categorical | Credit history |
| verw | Categorical | Purpose |
| hoehe | Numerical | Credit amount |
| sparkont | Categorical | Savings account / bonds |
| beszeit | Categorical | Present employment since |
| rate | Numerical | Installment rate (% of disposable income) |
| famges | Categorical | Personal status and sex |
| buerge | Categorical | Other debtors / guarantors |
| wohnzeit | Numerical | Present residence since |
| verm | Categorical | Property |
| alter | Numerical | Age in years |
| weitkred | Categorical | Other instalment plans |
| wohn | Categorical | Housing |
| bishkred | Numerical | Number of existing credits |
| beruf | Categorical | Job |
| pers | Numerical | Number of dependants |
| telef | Categorical | Telephone |
| gastarb | Categorical | Foreign worker |
| **kredit** | **Target** | **0 = good, 1 = bad (recoded)** |

> Refer to the South German Credit codebook for full coding details.
